# Notebook 2: Standalone High-Speed XADC DMA Acquisition & Hardware Triggering

This notebook demonstrates how to load the hardware overlay using **`OscilloscopeOverlay`**, configure the FPGA **`HardwareTrigger`** registers, and capture 1 MSPS analog streams direct to DDR memory.

## 1. Load Hardware Overlay
Instantiate `OscilloscopeOverlay()`. It automatically identifies the board (PYNQ-Z2) and fetches the pinned release binaries from GitHub Releases.

In [ ]:
from pynq_oscilloscope import OscilloscopeOverlay

# Auto-detect board and load overlay (or pass local path './pynq_z2.bit')
ol = OscilloscopeOverlay()
print("Oscilloscope Overlay loaded successfully!")
print("Active Trigger Configuration:", ol.trigger)

## 2. Configure Hardware Trigger Registers
Configure the FPGA trigger unit (`axis_trigger_unit_0` @ `0x43C10000`) for a Rising-edge trigger at 1.65V with Auto timeout.

In [ ]:
# Set Hardware Trigger: Rising Edge @ 1.65V in Auto Mode
ol.trigger.configure(mode="Auto", edge="Rising", threshold_volts=1.65, timeout_ms=50.0)
print(f"Hardware Trigger Threshold: {ol.trigger.get_threshold():.2f} V")

## 3. High-Speed 1 MSPS DMA Capture
Trigger the acquisition. Sample `[0]` in the returned array is guaranteed to be hardware-aligned to the trigger edge!

In [ ]:
# Capture 2,048 samples (2.048 ms of 1 MSPS data)
voltages = ol.capture()
print(f"Captured {len(voltages)} samples successfully!")
print(f"Sample Voltage Range: Min = {voltages.min():.2f} V, Max = {voltages.max():.2f} V")

## 4. Waveform Visualization with Matplotlib
Plot the captured waveform.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=100)
ax.plot(voltages[:500], color="#00A389", linewidth=1.8, label="A0 Channel (Hardware Triggered)")
ax.axhline(1.65, color="#FFA500", linestyle="--", label="Trigger Threshold (1.65V)")

ax.set_title("Hardware-Triggered 1 MSPS XADC Stream Capture", fontsize=12, fontweight="bold")
ax.set_xlabel("Time (Microseconds / Samples)", fontsize=10)
ax.set_ylabel("Measured Voltage (V)", fontsize=10)
ax.set_ylim(0, 3.5)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()